# Task 15: Enterprise LLM Gateway with Dynamic Rate-Limiting and Fallback Governance


## Objective

Build a compact FastAPI gateway design with Redis token-bucket rate limiting, provider fallback, and Prometheus metrics.


## Short Theory

The gateway sits between clients and LLM providers. Redis stores rate-limit state, fallback routes failed primary requests to a secondary provider, and Prometheus exposes metrics.


## Step 1: Imports


In [1]:
# pip install fastapi uvicorn redis prometheus-client httpx
from fastapi import FastAPI, HTTPException
from prometheus_client import Counter, Histogram
import time


## Step 2: Gateway and Metrics


In [2]:
app = FastAPI()
requests = Counter("llm_requests_total", "LLM requests")
latency = Histogram("llm_latency_seconds", "LLM latency")


## Step 3: Token-Bucket Logic


In [3]:
buckets = {}
def allow(key, limit=5, refill=1):
    now = time.time()
    tokens, last = buckets.get(key, (limit, now))
    tokens = min(limit, tokens + (now-last)*refill)
    if tokens < 1:
        return False
    buckets[key] = (tokens-1, now)
    return True


## Step 4: Provider Fallback


In [4]:
def call_provider(prompt, primary_ok=True):
    if primary_ok:
        return {"provider":"primary", "text":"Primary model response"}
    return {"provider":"secondary", "text":"Fallback model response"}

@app.post("/generate")
def generate(prompt: str):
    requests.inc()
    if not allow("demo-client"):
        raise HTTPException(429, "Rate limit exceeded")
    start = time.time()
    result = call_provider(prompt, primary_ok=False)
    latency.observe(time.time()-start)
    return result


## Step 5: Run


In [5]:
# uvicorn this_notebook:app --reload
print("LLM gateway configured with rate limiting and fallback.")


LLM gateway configured with rate limiting and fallback.


## Small Experiment

Fallback Experiment


In [6]:
print(call_provider("Hello", primary_ok=False))


{'provider': 'secondary', 'text': 'Fallback model response'}


## Conclusion

Demonstrated the gateway control plane with rate limiting, provider fallback, latency tracking, and Prometheus metrics. Replace the in-memory bucket with Redis for the required distributed deployment.
